# Phase 5 Demo: Redis-Based Paper Trading with Contract Resolution

This notebook demonstrates the Phase 5 capabilities:
- **Contract Symbol Resolution** (auto-detection of VN30F1M/F2M actual codes)
- **RedisPaperTradingEngine** (orchestrates all trading components)
- **Redis Streaming** (real-time market data via Redis Pub/Sub)
- **Results Export** (JSON serialization and summary reports)
- **Performance Benchmarking** (throughput and latency testing)

## Prerequisites

```bash
# Start Redis server
redis-server

# Verify Redis is running
redis-cli ping  # Should return "PONG"
```

In [ ]:
# Imports
import sys
import time
import json
from decimal import Decimal
from datetime import datetime, date
from pathlib import Path
import redis

# Add project root to path
sys.path.insert(0, '..')

from utils.contract_resolver import ContractSymbolResolver
from paper_trading.engine import RedisPaperTradingEngine
from paper_trading.results import PaperTradingResults
from tools.redis_publisher import RedisMarketDataPublisher

print("✅ All imports successful")

## Part 1: Contract Symbol Resolution

The most critical feature of Phase 5 is automatic resolution of contract symbols. Vietnamese futures data providers use exact month codes (VN30F2510, VN30F2511) that follow expiration rules, not informal symbols (VN30F1M, VN30F2M).

**Wrong ticker symbols have caused "many many bugs" in live systems.**

In [ ]:
# Create contract resolver with auto-detection
resolver = ContractSymbolResolver()

print("🔍 Contract Symbol Resolution (Auto-Detection Mode)\n")

# Resolve F1 and F2 contracts
f1_code = resolver.resolve('VN30F1M')
f2_code = resolver.resolve('VN30F2M')

print(f"VN30F1M → {f1_code}")
print(f"VN30F2M → {f2_code}")

# Get detailed information
summary = resolver.get_resolution_summary(['VN30F1M', 'VN30F2M'])

print("\n📅 Expiration Details:\n")
for informal, info in summary.items():
    code = info['code']
    expiration = info['expiration']
    days = info['days_to_expiry']
    
    expiry_str = expiration.strftime('%b %d, %Y (Third Thursday)')
    print(f"{informal}:")
    print(f"  Contract Code: {code}")
    print(f"  Expires: {expiry_str}")
    print(f"  Days to Expiry: {days} days")
    print()

### Manual Override Mode

For historical data testing, you can manually specify contract mappings:

In [ ]:
# Create resolver with manual mappings (for historical testing)
historical_mappings = {
    'VN30F1M': 'VN30F2202',  # February 2022
    'VN30F2M': 'VN30F2203'   # March 2022
}

manual_resolver = ContractSymbolResolver(
    auto_detect=False,
    contract_mappings=historical_mappings
)

print("🔧 Manual Override Mode (Historical Data)\n")
print(f"VN30F1M → {manual_resolver.resolve('VN30F1M')}")
print(f"VN30F2M → {manual_resolver.resolve('VN30F2M')}")

# Get expiration dates for historical contracts
f1_exp = manual_resolver.get_expiration_date('VN30F2202')
f2_exp = manual_resolver.get_expiration_date('VN30F2203')

print(f"\nVN30F2202 expired: {f1_exp.strftime('%b %d, %Y')}")
print(f"VN30F2203 expired: {f2_exp.strftime('%b %d, %Y')}")

### Auto-Detection from CSV Data

For historical data with actual contract codes in the CSV, use the `from_csv()` method to automatically detect F1M and F2M:

In [ ]:
# Auto-detect contracts from CSV data
csv_path = '../data/sample/merged_is_data_1day.csv'

if Path(csv_path).exists():
    # Create resolver by scanning CSV
    csv_resolver = ContractSymbolResolver.from_csv(csv_path)
    
    print("📊 Auto-Detection from CSV\n")
    print(f"CSV: {Path(csv_path).name}")
    print(f"VN30F1M → {csv_resolver.resolve('VN30F1M')}")
    print(f"VN30F2M → {csv_resolver.resolve('VN30F2M')}")
    
    # Get expiration dates
    f1_code = csv_resolver.resolve('VN30F1M')
    f2_code = csv_resolver.resolve('VN30F2M')
    
    f1_exp = csv_resolver.get_expiration_date(f1_code)
    f2_exp = csv_resolver.get_expiration_date(f2_code)
    
    print(f"\n{f1_code} expires: {f1_exp.strftime('%b %d, %Y')}")
    print(f"{f2_code} expires: {f2_exp.strftime('%b %d, %Y')}")
    print("\n✅ Auto-detection complete - ready for backtesting!")
else:
    print(f"⚠️  Sample data not found: {csv_path}")
    print("   Run: python tools/create_sample_data_from_merged.py")

## Part 2: Redis Connection Test

Verify Redis is available before starting paper trading.

In [ ]:
# Test Redis connection
try:
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)
    response = redis_client.ping()
    print(f"✅ Redis connection successful: {response}")
    
    # Get Redis info
    info = redis_client.info('server')
    print(f"\nRedis Server Info:")
    print(f"  Version: {info.get('redis_version')}")
    print(f"  OS: {info.get('os')}")
    
except redis.ConnectionError as e:
    print(f"❌ Redis connection failed: {e}")
    print("\nMake sure Redis is running:")
    print("  redis-server")

## Part 3: Simple Paper Trading Session

Run a basic paper trading session with manual market data.

In [ ]:
# Create Redis publisher
publisher = RedisMarketDataPublisher(
    redis_host='localhost',
    redis_port=6379,
    channel_prefix='test_market'
)

if not publisher.connect():
    raise RuntimeError("Failed to connect publisher to Redis")

print("✅ Publisher connected to Redis")

In [ ]:
# Create paper trading engine
engine = RedisPaperTradingEngine(
    initial_capital=Decimal('500000'),
    step=Decimal('2.9'),
    redis_host='localhost',
    redis_port=6379,
    channel_prefix='test_market',
    contracts=['VN30F2202', 'VN30F2203'],  # Using resolved codes
    update_interval_seconds=2  # Short interval for demo
)

print("✅ Engine created")
print(f"   Initial capital: {engine.initial_capital:,}")
print(f"   Strategy step: {engine.step}")
print(f"   Contracts: {engine.contracts}")

In [ ]:
# Start engine (non-blocking)
engine.start()
print("🚀 Engine started")
print(f"   Status: {'RUNNING' if engine.running else 'STOPPED'}")

# Give it a moment to initialize
time.sleep(0.5)

In [ ]:
# Publish market data
print("📤 Publishing market data...\n")

base_price = 1250.0

for i in range(30):
    # Vary price slightly
    price_offset = (i % 10) * 0.5
    
    for contract in ['VN30F2202', 'VN30F2203']:
        # F2 typically trades at a premium
        contract_offset = 5.0 if contract == 'VN30F2203' else 0.0
        price = base_price + price_offset + contract_offset
        
        message_data = {
            'timestamp': datetime.now().isoformat(),
            'contract': contract,
            'price': price,
            'bid': price - 1.0,
            'ask': price + 1.0,
            'spread': 2.0
        }
        publisher.publish_message(contract, message_data)
    
    if i % 10 == 0:
        print(f"  Published {i+1}/30 batches...")
    
    time.sleep(0.1)

print("\n✅ Market data published")

# Wait for processing
print("⏳ Processing events...")
time.sleep(3)
print("✅ Processing complete")

In [ ]:
# Get real-time summary
summary = engine.get_summary()

print("📊 Real-Time Summary\n")
print(f"Status: {summary['status'].upper()}")
print(f"\nPortfolio:")
print(f"  Initial NAV:   {summary['initial_nav']:>12,.0f}")
print(f"  Current NAV:   {summary['current_nav']:>12,.0f}")
print(f"  PnL:           {summary['pnl']:>12,.0f} ({summary['return_pct']:+.2f}%)")

print(f"\nTrading:")
print(f"  Total Trades:  {summary['total_trades']:>12}")
print(f"  Total Fees:    {summary['total_fees']:>12,.0f}")

print(f"\nRedis:")
print(f"  Messages:      {summary['messages_processed']:>12}")
print(f"  Latency:       {summary['redis_latency_ms']:>12.2f} ms")

In [ ]:
# Stop engine and get results
print("🛑 Stopping engine...")
results = engine.stop()
print("✅ Engine stopped\n")

# Print detailed results
results.print_summary()

## Part 4: Results Export and Analysis

In [ ]:
# Export results to JSON
output_path = Path('../results/demo_session.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

results.to_json(str(output_path))
print(f"✅ Results exported to: {output_path}")

# Load and display JSON structure
with open(output_path, 'r') as f:
    data = json.load(f)

print(f"\n📄 JSON Structure:")
for key in data.keys():
    print(f"  - {key}")

In [ ]:
# Analyze performance metrics
print("📈 Performance Analysis\n")

print(f"Session Duration: {results.duration_seconds:.1f}s ({results.duration_seconds/60:.1f} minutes)")
print(f"\nReturns:")
print(f"  HPR:           {results.hpr:>8.2%}")
print(f"  Sharpe Ratio:  {results.sharpe_ratio:>8.2f}")
print(f"  Sortino Ratio: {results.sortino_ratio:>8.2f}")
print(f"  Max Drawdown:  {results.max_drawdown:>8.2%}")

print(f"\nTrading Activity:")
print(f"  Total Trades:  {results.total_trades:>8}")
print(f"  Buy Trades:    {results.buy_trades:>8}")
print(f"  Sell Trades:   {results.sell_trades:>8}")
print(f"  Total Fees:    {results.total_fees:>8,.0f} VND")

print(f"\nRedis Performance:")
print(f"  Messages:      {results.messages_received:>8}")
print(f"  Processed:     {results.messages_processed:>8}")
print(f"  Success Rate:  {results.messages_processed/results.messages_received*100:>7.1f}%")
print(f"  Avg Latency:   {results.avg_latency_ms:>7.2f} ms")
print(f"  Reconnects:    {results.reconnect_count:>8}")

## Part 5: CSV Playback Integration

Test with historical CSV data published through Redis.

In [ ]:
# Check for sample data
sample_path = Path('../data/sample/merged_is_data_1day.csv')

if sample_path.exists():
    print(f"✅ Sample data found: {sample_path.name}")
    print(f"   Size: {sample_path.stat().st_size / 1024:.1f} KB")
else:
    print(f"❌ Sample data not found: {sample_path}")
    print("\nPlease create sample data first:")
    print("  python tools/create_sample_data_from_merged.py")

In [ ]:
# Load CSV into publisher (if available)
if sample_path.exists():
    # Create new publisher for CSV playback
    csv_publisher = RedisMarketDataPublisher(
        redis_host='localhost',
        redis_port=6379,
        channel_prefix='historical'
    )
    
    if csv_publisher.connect():
        print("✅ CSV Publisher connected")
        
        # Load CSV
        csv_publisher.load_csv(str(sample_path))
        print(f"✅ CSV loaded: {len(csv_publisher.data)} rows")
        
        # Show contracts
        contracts = csv_publisher.data['tickersymbol'].unique()
        print(f"   Contracts: {', '.join(contracts)}")
    else:
        print("❌ Failed to connect CSV publisher")

In [ ]:
# Run paper trading with CSV playback (if data available)
if sample_path.exists() and csv_publisher.data is not None:
    import threading
    
    # Start publishing in background thread
    def publish_csv():
        csv_publisher.start_publishing(rate_hz=50, loop=False)
    
    publisher_thread = threading.Thread(target=publish_csv)
    publisher_thread.daemon = True
    
    print("🚀 Starting CSV playback...\n")
    publisher_thread.start()
    
    # Wait for publisher to start
    time.sleep(0.5)
    
    # Create engine with historical contracts
    csv_engine = RedisPaperTradingEngine(
        initial_capital=Decimal('500000'),
        step=Decimal('2.9'),
        redis_host='localhost',
        redis_port=6379,
        channel_prefix='historical',
        contracts=['VN30F2202', 'VN30F2203'],
        update_interval_seconds=15
    )
    
    # Run for limited time
    print("▶️  Running paper trading for 10 seconds...\n")
    csv_results = csv_engine.run(duration_seconds=10)
    
    print("\n✅ CSV playback complete\n")
    
    # Show results
    print(f"Performance Summary:")
    print(f"  Messages:  {csv_results.messages_received}")
    print(f"  NAV:       {csv_results.final_nav:,.0f}")
    print(f"  HPR:       {csv_results.hpr:+.2%}")
    print(f"  Trades:    {csv_results.total_trades}")

## Part 6: Performance Benchmarking

Measure throughput and latency under load.

In [ ]:
# Throughput benchmark
print("⚡ Throughput Benchmark\n")

# Create benchmark publisher
bench_publisher = RedisMarketDataPublisher(
    redis_host='localhost',
    redis_port=6379,
    channel_prefix='benchmark'
)

if bench_publisher.connect():
    # Create benchmark engine
    bench_engine = RedisPaperTradingEngine(
        initial_capital=Decimal('500000'),
        step=Decimal('2.9'),
        redis_host='localhost',
        redis_port=6379,
        channel_prefix='benchmark',
        contracts=['VN30F2202'],
        update_interval_seconds=15
    )
    
    bench_engine.start()
    time.sleep(0.3)
    
    # Publish large number of messages
    message_count = 200
    start_time = time.time()
    
    print(f"Publishing {message_count} messages...")
    for i in range(message_count):
        message_data = {
            'timestamp': datetime.now().isoformat(),
            'contract': 'VN30F2202',
            'price': 1250.0 + (i % 50) * 0.1,
            'bid': 1249.0,
            'ask': 1251.0
        }
        bench_publisher.publish_message('VN30F2202', message_data)
    
    publish_time = time.time() - start_time
    
    # Wait for processing
    time.sleep(2)
    
    # Get results
    bench_results = bench_engine.stop()
    
    # Calculate metrics
    throughput = bench_results.messages_processed / bench_results.duration_seconds
    processing_rate = bench_results.messages_processed / message_count
    
    print(f"\n{'='*60}")
    print("BENCHMARK RESULTS")
    print(f"{'='*60}")
    print(f"Messages published:  {message_count:>8}")
    print(f"Messages processed:  {bench_results.messages_processed:>8}")
    print(f"Processing rate:     {processing_rate:>7.1%}")
    print(f"Publish time:        {publish_time:>7.2f}s")
    print(f"Throughput:          {throughput:>7.1f} msg/s")
    print(f"Avg latency:         {bench_results.avg_latency_ms:>7.2f} ms")
    print(f"{'='*60}")
    
    # Performance assertions
    if processing_rate >= 0.9:
        print("\n✅ Processing rate: EXCELLENT (≥90%)")
    else:
        print(f"\n⚠️  Processing rate: {processing_rate:.1%} (target: ≥90%)")
    
    if throughput >= 50:
        print("✅ Throughput: EXCELLENT (≥50 msg/s)")
    else:
        print(f"⚠️  Throughput: {throughput:.1f} msg/s (target: ≥50 msg/s)")
    
    if bench_results.avg_latency_ms > 0 and bench_results.avg_latency_ms < 50:
        print("✅ Latency: EXCELLENT (<50ms)")
    elif bench_results.avg_latency_ms > 0:
        print(f"⚠️  Latency: {bench_results.avg_latency_ms:.2f}ms (target: <50ms)")
    
    bench_publisher.disconnect()
else:
    print("❌ Failed to connect benchmark publisher")

## Part 7: Blocking Run Mode

Demonstrate the blocking `run()` method for specific durations.

In [ ]:
print("⏱️  Blocking Run Mode Demo\n")

# Create publisher thread
blocking_publisher = RedisMarketDataPublisher(
    redis_host='localhost',
    redis_port=6379,
    channel_prefix='blocking_test'
)

if blocking_publisher.connect():
    import threading
    
    # Publish in background
    def publish_continuous():
        for i in range(100):
            message_data = {
                'timestamp': datetime.now().isoformat(),
                'contract': 'VN30F2202',
                'price': 1250.0 + (i % 20) * 0.2,
                'bid': 1249.0,
                'ask': 1251.0
            }
            blocking_publisher.publish_message('VN30F2202', message_data)
            time.sleep(0.1)
    
    thread = threading.Thread(target=publish_continuous)
    thread.daemon = True
    thread.start()
    
    # Create engine
    blocking_engine = RedisPaperTradingEngine(
        initial_capital=Decimal('500000'),
        step=Decimal('2.9'),
        redis_host='localhost',
        redis_port=6379,
        channel_prefix='blocking_test',
        contracts=['VN30F2202'],
        update_interval_seconds=15
    )
    
    # Run for 5 seconds (blocking call)
    print("Running engine for 5 seconds (blocking)...\n")
    start = time.time()
    blocking_results = blocking_engine.run(duration_seconds=5)
    elapsed = time.time() - start
    
    print(f"\n✅ Completed in {elapsed:.1f}s\n")
    
    print("Results:")
    print(f"  Duration:      {blocking_results.duration_seconds:.1f}s")
    print(f"  Messages:      {blocking_results.messages_processed}")
    print(f"  Final NAV:     {blocking_results.final_nav:,.0f}")
    print(f"  Trades:        {blocking_results.total_trades}")
    
    blocking_publisher.disconnect()
else:
    print("❌ Failed to connect publisher")

## Part 8: Cleanup

In [ ]:
print("🧹 Cleaning up...\n")

# Close publisher
if publisher:
    publisher.disconnect()
    print("✅ Publisher disconnected")

# Close Redis client
if redis_client:
    redis_client.close()
    print("✅ Redis client closed")

print("\n✨ Cleanup complete")

## Summary

This notebook demonstrated Phase 5 capabilities:

### ✅ Key Features

1. **Contract Symbol Resolution** - Auto-detection of F1/F2 actual codes
2. **Paper Trading Engine** - Full orchestration of trading components
3. **Redis Integration** - Real-time market data streaming
4. **Results Export** - JSON serialization and detailed reporting
5. **CSV Playback** - Historical data testing through Redis
6. **Performance Benchmarking** - Throughput and latency measurement

### 🎯 Production Safety

- **Contract resolution prevents ticker symbol bugs** ("many many bugs" in live systems)
- **Safety confirmation prompts** before trading (CLI runner)
- **Two-parameter switch** for local→production deployment
- **Comprehensive logging** and error handling

### 📊 Performance Targets

- **Throughput**: ≥50 msg/s
- **Latency**: <50ms average
- **Processing rate**: ≥90%
- **Reliability**: Auto-reconnect, graceful error recovery

### 🚀 Next Steps

- Run CLI runner for production-like testing
- Test with multi-day historical data (with rollovers)
- Compare results with ground truth logs
- Deploy to production with live data feed

### 📚 Additional Resources

- Phase 5 Spec: `internal-docs/paper-trading-phase-5-spec.md`
- Completion Report: `markdown-docs/phase-5-completion-report.md`
- CLI Runner: `python -m paper_trading.runner --help`
- Integration Tests: `tests/integration/test_paper_trading_integration.py`